# Naturalness Scores for double mutants

**Goal:** Evaluate different "double mutant scoring" strategies.

**Background:** ESMC based scoring double mutants with comparison to the few multi-mutant datasets in ProteinGym


In [2]:
import numpy  as np
import pandas as pd
import dask.dataframe as dd
# from dask.distributed import Client; Client()

###############################################################################
# 1.  Olson DMS score lookup
###############################################################################
act_df = (pd.read_csv('EvCoupling/SPG1/SPG1_STRSG_Olson_2014.csv',
                      usecols=['mutant', 'DMS_score'])
            .assign(seq_id=lambda s: s['mutant'].str.replace(':', '_',
                                                             regex=False))
            .drop(columns='mutant'))
print(act_df['DMS_score'].quantile(0.99))
dms_lookup = act_df.set_index('seq_id')['DMS_score'].to_dict()

###############################################################################
# 2.  Read only the needed columns from the big file
###############################################################################
big = dd.read_csv(
        'EvCoupling/SPG1/SPG1_STRSG_Olson_2014_naturalness_600m_doubles.csv',
        usecols=['seq_id', 'probability', 'base_seq_id'],
        dtype={'seq_id': 'object',
               'probability': 'float64',
               'base_seq_id': 'object'},
        blocksize='16 MiB')
print(len(act_df))
n_rows = big.shape[0].compute()
print(n_rows)
###############################################################################
# 3.  WT look‑ups
###############################################################################
wt_pd = (big.query('base_seq_id == "WT"')[['seq_id', 'probability']]
           .drop_duplicates(subset='seq_id')
           .compute()
           .rename(columns={'probability': 'wt_prob'}))

seq_lookup = wt_pd.set_index('seq_id')['wt_prob'].to_dict()

wt_locus_df = wt_pd[wt_pd.seq_id.str[0] == wt_pd.seq_id.str[-1]].copy()
wt_locus_df['locus'] = wt_locus_df.seq_id.str.slice(1, -1).astype(int)
locus_lookup = wt_locus_df.set_index('locus')['wt_prob'].to_dict()

###############################################################################
# 4.  Allowed set of residue letters (lower‑case for easy matching)
###############################################################################
allowed_residues = set('asdfghklqwertyipcvnm')

###############################################################################
# 5.  Partition function
###############################################################################
def add_all(part, seq_lut, loc_lut, dms_lut, allowed):
    # first / second mutation strings
    part['first_mut']  = part['base_seq_id']
    part['second_mut'] = part['seq_id']
    part['double_mut'] = part['base_seq_id'] +'_'+ part['seq_id']
    # ─────────────  FILTER on second_mut last character  ─────────────
    mask = part['second_mut'].str[-1].str.lower().isin(allowed)
    part = part[mask].copy()

    # ---- WT probability for the whole seq_id
    part['wt_prob'] = part['seq_id'].map(seq_lut)

    # ---- numeric loci
    part['first_locus']  = pd.to_numeric(
                              part['first_mut'].str.slice(1, -1), errors='coerce')
    part['second_locus'] = pd.to_numeric(
                              part['second_mut'].str.slice(1, -1), errors='coerce')

    # ---- probabilities for single mutants
    part['first_mut_prob']   = part['first_mut'].map(seq_lut)
    part['second_mut_prob']  = part['second_mut'].map(seq_lut)

    # ---- WT probabilities per locus
    part['first_mut_wt_prob']  = part['first_locus'] .map(loc_lut)
    part['second_mut_wt_prob'] = part['second_locus'].map(loc_lut)

    # ---- reversion score
    part['reversion_score'] = part['probability'] - part['wt_prob']

    # ---- DMS score
    part['DMS_score'] = part['double_mut'].map(dms_lut)

    # ---- double‑mutant naturalness and its log
    part['double_mutant_naturalness'] = (
        (part['first_mut_prob']  / part['first_mut_wt_prob']) *
        (part['second_mut_prob'] / part['second_mut_wt_prob'])
    )
    part['log_double_mutant_naturalness'] = np.log(
        part['double_mutant_naturalness'])

    # ---- drop rows that have NaN in log_double_mutant_naturalness
    part = part.dropna(subset=['log_double_mutant_naturalness'])

    # ---- keep only the requested columns
    return part[['double_mut','DMS_score',
                 'log_double_mutant_naturalness',
                 'reversion_score']]

###############################################################################
# 6.  Apply the function
###############################################################################
meta = {
    'double_mut': 'object',
    'DMS_score'                   : 'float64',
        'log_double_mutant_naturalness': 'float64',
        'reversion_score'             : 'float64'}

df = big.map_partitions(add_all,
                        seq_lut=seq_lookup,
                        loc_lut=locus_lookup,
                        dms_lut=dms_lookup,
                        allowed=allowed_residues,
                        meta=meta)

###############################################################################
# 7.  Persist / inspect / write
###############################################################################
df = df.persist()
print(df['DMS_score'].quantile(0.99))
print(df.head())

# df.to_parquet('final_scores.parquet')
# result = df.compute()

1.672187926605294
536962
125856192
<dask_expr.expr.Scalar: expr=SeriesQuantile(frame=FromGraph(ce8ecdc)['DMS_score'], q=0.99), dtype=float64>
  double_mut  DMS_score  log_double_mutant_naturalness  reversion_score
4   T30E_M1L        NaN                     -11.378037     0.000000e+00
5   T30E_M1A        NaN                     -14.348061     8.344650e-07
6   T30E_M1G        NaN                     -14.188847     8.344650e-07
7   T30E_M1V        NaN                     -12.067026     0.000000e+00
8   T30E_M1S        NaN                     -13.723182     2.861023e-06


In [ ]:
###############################################################################
import matplotlib.pyplot as plt
import math
###############################################################################
# 8.  Keep only rows with a real DMS_score  (still lazy)
###############################################################################
cols = ['reversion_score', 'log_double_mutant_naturalness', 'DMS_score']

if 'dask' in str(type(df)):          # crude but fast “is this a dask frame?”
    plot_df = df[cols].dropna().compute()
else:                                # already pandas
    plot_df = df[cols].dropna()
print(len(plot_df))
print(len(df))
def top_n_thresholds(df, column, n_list):
    """
    For each N in n_list return the value X so that the top‑N rows of column
    have column ≥ X.  Works for either Dask or Pandas frames.
    """
    is_dask = 'dask' in str(type(df))
    # total number of rows ­— needs a single scalar compute() for Dask
    n_total = df.shape[0].compute() if is_dask else len(df)

    thresholds = {}
    for n in n_list:
        if n > n_total:
            thresholds[n] = math.nan         # (or raise / skip)
            continue

        q = max(0, 1 - n / n_total)         # e.g. N=50k in 5 M rows → q=0.99
        if is_dask:
            thresholds[n] = df[column].quantile(q).compute()
        else:
            thresholds[n] = df[column].quantile(q)

    return thresholds
# top_ns    = [100_000, 250_000, 500_000, 1_000_000,5_000_000,10_000_000]          # add / remove as you like
# cut_cols  = ['reversion_score', 'log_double_mutant_naturalness']
# rev_thr  = top_n_thresholds(df, cut_cols[0], top_ns)     # dict {N: value}
# log_thr  = top_n_thresholds(df, cut_cols[1], top_ns)
top_pcts = [
            0.001,
            0.005,
            0.01,            # 1.0  %
            0.02,            # 2.0  %
            0.05,            # 5.0  %
            0.5,
            0.95,
            0.98,
            0.99
            ]           

###############################################################################
# 1.  A helper that works for Pandas or Dask
###############################################################################
def top_pct_thresholds(df, column, pct_list):
    """
    For each p in pct_list (0 < p ≤ 1) return the value X so that the
    top‑p fraction of *column* has column ≥ X.
    Works for either Dask or Pandas DataFrames.
    """
    is_dask = 'dask' in str(type(df))
    th = {}
    for p in pct_list:
        q = 1.0 - p                         # e.g. top‑1 %  → quantile 0.99
        if is_dask and p <= 0.5:
            th[p] = df[column].quantile(q).compute()
            per_ddf   = df[df[column] >= th[p]]
            n_matches = per_ddf.shape[0].compute()   # <– THIS is the #rows
            print(f'this is {p*100} percentile of {column}: {n_matches:,d} rows')
        else:
            th[p] = df[column].quantile(q)
    return th
def intersection_counts(df, colA, colB, thrA, thrB):
    """
    Count rows that are simultaneously above the two sets of thresholds.

    Parameters
    ----------
    df     : pandas.DataFrame or dask.DataFrame
    colA   : str   – first column name
    colB   : str   – second column name
    thrA   : dict  – {p : threshold_value} for colA
    thrB   : dict  – {p : threshold_value} for colB

    Returns
    -------
    pandas.DataFrame  (index = p for colA, columns = p for colB)
    """
    is_dask = 'dask' in str(type(df))
    # create the empty result table
    out = pd.DataFrame(index=thrA.keys(), columns=thrB.keys(), dtype='int64')

    # pre‑build the boolean Series once so that we do not evaluate
    # the same “>= threshold” expression again and again
    maskA = {p: (df[colA] >= thr) for p, thr in thrA.items()}
    maskB = {p: (df[colB] >= thr) for p, thr in thrB.items()}

    for pA, mA in maskA.items():
        for pB, mB in maskB.items():
            # intersection
            both = mA & mB
            # turn the boolean mask into the #rows
            n = both.sum().compute() if is_dask else both.sum()
            out.loc[pA, pB] = int(n)

    return out
###############################################################################
# 2.  Get the thresholds
###############################################################################
cut_cols = ['reversion_score', 'log_double_mutant_naturalness']
rev_thr  = top_pct_thresholds(df, cut_cols[0], top_pcts)     # dict {p: value}
log_thr  = top_pct_thresholds(df, cut_cols[1], top_pcts)
combo_cnts = intersection_counts(df,
                                 'reversion_score',
                                 'log_double_mutant_naturalness',
                                 rev_thr,
                                 log_thr)
print(combo_cnts)
###############################################################################
# 9‑A.  Classic scatter (Matplotlib)  ─────────────────────────────────────────
#      Suitable for ≤ a few million points.  Downsample if needed.


# If the data set is still too big for RAM, sample before compute:
# plot_df = plot_df.sample(frac=0.1, random_state=42)   # 10 % random sample
quantile = 0.999
# ────────────────────────────────────────────────────────────────────────────
# NEW ➜ 1.  Work out the 99th‑percentile threshold for DMS_score
dms_thr = plot_df['DMS_score'].quantile(quantile)   # one float
# ────────────────────────────────────────────────────────────────────────────
print(dms_thr)
plot_pd = plot_df      # three columns, so usually small enough

# ────────────────────────────────────────────────────────────────────────────
# NEW ➜ 2.  Boolean masks for “top n %” vs “the rest”
mask_top  =  plot_pd['DMS_score'] >= dms_thr
mask_rest = ~mask_top
# ────────────────────────────────────────────────────────────────────────────

fig, ax = plt.subplots(figsize=(15, 12))

# ────────────────────────────────────────────────────────────────────────────
# NEW ➜ 3‑a.  First plot **the rest** (background) — same as before
sc = ax.scatter(plot_pd.loc[mask_rest, 'reversion_score'],
                plot_pd.loc[mask_rest, 'log_double_mutant_naturalness'],
                c = plot_pd.loc[mask_rest, 'DMS_score'],
                cmap='viridis',
                s=30, alpha=0.6, linewidths=0)

# NEW ➜ 3‑b.  Overlay the **top 1 %** in red, slightly larger markers
ax.scatter(plot_pd.loc[mask_top, 'reversion_score'],
           plot_pd.loc[mask_top, 'log_double_mutant_naturalness'],
           c='red', s=90, alpha=0.9, edgecolor = 'black', linewidth=.5,
           label=f'Top {(1-quantile)*100:.2f} % DMS_score')
# ────────────────────────────────────────────────────────────────────────────
# for n in top_ns:
#     # vertical for reversion_score
#     if not math.isnan(rev_thr[n]):
#         ax.axvline(x=rev_thr[n], color='darkorange', ls='--', lw=1.4,
#         )

#     # horizontal for log_double_mutant_naturalness
#     if not math.isnan(log_thr[n]):
#         ax.axhline(y=log_thr[n], color='black', ls=':',  lw=1.4,
#                   )
for p in top_pcts:
    # vertical for reversion_score
    ax.axvline(x=rev_thr[p],
               color='darkorange', ls='--', lw=1.4,
            #    label=f'top {p*100:.1f}% (reversion)' if p==top_pcts[0] else None
               )

    # horizontal for log_double_mutant_naturalness
    ax.axhline(y=log_thr[p],
               color='black', ls=':',  lw=1.4,
            #    label=f'top {p*100:.1f}% (naturalness)' if p==top_pcts[0] else None
               )
ax.set_xscale('symlog', linthresh=1e-7)
ax.set_xlabel('Reversion score (symlog)')
ax.set_ylabel('log double‑mutant naturalness')
ax.set_title(f'Top {[100 * i for i in top_pcts]}')
ax.grid(True, which='both', ls='--', lw=0.5)
cbar = plt.colorbar(sc, ax=ax, label='DMS score')
ax.legend()                              # NEW ➜ legend for red highlights

plt.tight_layout()
plt.show()
